# 05c — Unity AI Gateway: LLM Model Service + Inference Table

**Jackson and Johnson HR Digital**

Creates (or updates) a governed **Unity AI Gateway model service** that the Hire Right agent and app
call for every LLM request, instead of hitting the legacy `databricks-gpt-5-4` serving endpoint.

| Item | Value |
|---|---|
| Model service | `{catalog}.{ai_gateway_schema}.{model_service}` (default `bldemos.hrd_digital.hire_right_llm`) |
| Routes to | `{destination_model}` (default `system.ai.databricks-gpt-5-4`, pay-per-token) |
| Inference table | `{catalog}.{ai_gateway_schema}.{model_service}_payload` (created on first request) |
| Usage | `system.ai_gateway.usage` (~1 hour latency) |
| Callers | agent SP `hire-right-agent-sp` and the app SP get `EXECUTE` on the service |

Callers hit `POST {host}/ai-gateway/mlflow/v1/chat/completions` with `model = "<catalog>.<schema>.<service>"`
(LangChain: `ChatDatabricks(endpoint=..., use_ai_gateway=True)`).

> Unity AI Gateway model services only route to **foundation models** (pay-per-token, provisioned
> throughput, external). The custom agent endpoint and the sklearn hiring model stay on Model Serving.

In [ ]:
dbutils.widgets.text("catalog",           "bldemos",                      "UC Catalog")
dbutils.widgets.text("ai_gateway_schema", "hrd_digital",                  "AI Gateway schema")
dbutils.widgets.text("model_service",     "hire_right_llm",               "Model service name")
dbutils.widgets.text("destination_model", "system.ai.databricks-gpt-5-4", "Destination foundation model")
dbutils.widgets.text("agent_sp_name",     "hire-right-agent-sp",          "Agent runtime service principal")
dbutils.widgets.text("app_name",          "hire-right-agent-app",         "Databricks App name")

In [ ]:
import requests
from databricks.sdk import WorkspaceClient

catalog           = dbutils.widgets.get("catalog")
gw_schema         = dbutils.widgets.get("ai_gateway_schema")
model_service     = dbutils.widgets.get("model_service")
destination_model = dbutils.widgets.get("destination_model")
agent_sp_name     = dbutils.widgets.get("agent_sp_name")
app_name          = dbutils.widgets.get("app_name")

service_full = f"{catalog}.{gw_schema}.{model_service}"

_ctx    = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host    = _ctx.apiUrl().get().rstrip("/")
headers = {"Authorization": f"Bearer {_ctx.apiToken().get()}", "Content-Type": "application/json"}
w = WorkspaceClient()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gw_schema} "
          f"COMMENT 'Hire Right — Unity AI Gateway model services and inference tables'")
print(f"Model service : {service_full}")
print(f"Destination   : {destination_model}")

## Create or update the model service

In [ ]:
MS_API = f"{host}/api/2.1/unity-catalog/model-services"

routing = {"destinations": [{
    "name":             destination_model,
    "destination_type": "DESTINATION_TYPE_PAY_PER_TOKEN_FOUNDATION_MODEL",
    "pay_per_token_config": {"model": f"models/{destination_model}"},
    "traffic_percentage": 100,
}]}
inference_table = {"parent": f"schemas/{catalog}.{gw_schema}", "table_name_prefix": model_service}
comment = (f"Hire Right Agent LLM via Unity AI Gateway. Routes to {destination_model}; "
           f"payloads logged to {catalog}.{gw_schema}.{model_service}_payload.")

_get = requests.get(f"{MS_API}/{service_full}", headers=headers, timeout=30)
if _get.status_code == 404:
    r = requests.post(MS_API, headers=headers, timeout=60,
                      params={"parent": f"schemas/{catalog}.{gw_schema}", "model_service_id": model_service},
                      json={"comment": comment, "config": {"routing": routing, "inference_table": inference_table}})
    r.raise_for_status()
    print(f"✅ Created model service {service_full}")
else:
    _get.raise_for_status()
    existing = _get.json()
    # Inference-table parent is immutable once created — only add it if missing.
    mask = ["comment", "config.routing.destinations"]
    cfg  = {"routing": routing}
    if not existing.get("config", {}).get("inference_table"):
        mask.append("config.inference_table")
        cfg["inference_table"] = inference_table
    r = requests.patch(f"{MS_API}/{service_full}", headers=headers, timeout=60,
                       params={"update_mask": ",".join(mask)},
                       json={"comment": comment, "config": cfg})
    r.raise_for_status()
    print(f"✅ Updated model service {service_full} ({', '.join(mask)})")

svc = r.json()
payload_table = svc["config"].get("inference_table", {}).get("table", "").removeprefix("tables/")
print(f"   Inference table : {payload_table}")
print(f"   API types       : {svc.get('supported_api_types')}")

## Grant callers EXECUTE

The agent endpoint runs as `hire-right-agent-sp` (OAuth M2M env vars set by notebook 06), and the app
calls the LLM as its own service principal. Both need `EXECUTE` on the model service plus
`USE CATALOG` / `USE SCHEMA` on its parent.

In [ ]:
principals = {}
_sp = next(iter(w.service_principals.list(filter=f'displayName eq "{agent_sp_name}"')), None)
if _sp:
    principals[agent_sp_name] = _sp.application_id
else:
    print(f"⚠️  Agent SP {agent_sp_name!r} not found — run scripts/setup_agent_sp.sh (notebook 06 re-grants)")
try:
    _app_sp = w.apps.get(name=app_name).service_principal_client_id
    if _app_sp:
        principals[f"app {app_name}"] = _app_sp
except Exception as e:
    print(f"⚠️  App {app_name!r} not found yet ({e}) — notebook 09 re-grants after deploy")

for label, app_id in principals.items():
    spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `{app_id}`")
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.{gw_schema} TO `{app_id}`")
    g = requests.patch(f"{host}/api/2.1/unity-catalog/permissions/model_service/{service_full}",
                       headers=headers, timeout=30,
                       json={"changes": [{"principal": app_id, "add": ["EXECUTE"]}]})
    g.raise_for_status()
    print(f"✓ {label} ({app_id}): USE CATALOG, USE SCHEMA, EXECUTE on {service_full}")

## Smoke test through the gateway

In [ ]:
r = requests.post(f"{host}/ai-gateway/mlflow/v1/chat/completions", headers=headers, timeout=120, json={
    "model": service_full,
    "messages": [{"role": "user", "content": "Reply with exactly: Hire Right gateway OK"}],
    "max_tokens": 20,
})
r.raise_for_status()
print("LLM reply:", r.json()["choices"][0]["message"]["content"])
print(f"\nPayloads → {payload_table} (async; table appears after the first flush)")
print("Usage    → system.ai_gateway.usage (~1h latency)")

dbutils.jobs.taskValues.set(key="llm_endpoint", value=service_full)